In [1]:
import os

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import scanpy as sc
import torch

import T_perturb
from T_perturb.Dataloaders.datamodule import CellGenDataModule
from T_perturb.Model.trainer import InSilicoPerturberGeneration
from T_perturb.src.utils import label_encoder
from T_perturb.tests.test_cellgen_training import dummy_dataset
from T_perturb.tests.test_countdecoder_training import dummy_cell_gene_matrix


/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
print("\nKernel directory path:")
print(os.path.dirname(sys.executable))


Kernel directory path:
/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/bin


In [3]:
if os.getcwd().split('/')[-1] != 'healthy_imm_expr':
    # set working directory to root of repository
    os.chdir('/lustre/scratch126/cellgen/team361/kl11/t_generative/')

In [4]:
tgt_vocab_size = 101  # +1 for padding token
num_samples = 100
num_genes = 100
max_seq_length = 50
n_total_tps = 2
num_samples = 100
batch_size = 4
pred_tps = [1, 2]
context_tps = [1, 2]
d_model = 12

genes_to_perturb = [70, 41]
perturbation_token = 0


In [5]:
src_counts = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
)
src_dataset = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
)
tgt_counts_dict = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
    total_time_steps=n_total_tps,
)
tgt_datasets = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
    total_time_steps=n_total_tps,
)

In [6]:
conditions = None
condition_keys = None
conditions_combined = None

In [7]:
if condition_keys is None:
    condition_keys = 'tmp_batch'
    # create a mock vector if there are no batch effect
    tmp_series = pd.DataFrame(
        {
            condition_keys: np.ones(num_samples),
        }
    )


In [8]:
if isinstance(condition_keys, str):
    condition_keys_ = [condition_keys]
else:
    condition_keys_ = condition_keys

if conditions is None:
    if condition_keys is not None:
        conditions_ = {}
        for cond in condition_keys_:
            conditions_[cond] = tmp_series[cond].unique().tolist()
    else:
        conditions_ = {}
else:
    conditions_ = conditions

if conditions_combined is None:
    if len(condition_keys_) > 1:
        tmp_series['conditions_combined'] = tmp_series[condition_keys].apply(
            lambda x: '_'.join(x), axis=1
        )
    else:
        tmp_series['conditions_combined'] = tmp_series[condition_keys]
    conditions_combined_ = tmp_series['conditions_combined'].unique().tolist()
else:
    conditions_combined_ = conditions_combined

condition_encodings = {
    cond: {k: v for k, v in zip(conditions_[cond], range(len(conditions_[cond])))}
    for cond in conditions_.keys()
}
conditions_combined_encodings = {
    k: v for k, v in zip(conditions_combined_, range(len(conditions_combined_)))
}



In [9]:
tgt_adata_tmp = sc.AnnData(X=tgt_counts_dict['tgt_h5ad_t1'].squeeze(), obs=tmp_series)

/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [10]:
if (condition_encodings is not None) and (condition_keys_ is not None):
    conditions = [
        label_encoder(
            tgt_adata_tmp,
            encoder=condition_encodings[condition_keys_[i]],
            condition_key=condition_keys_[i],
        )
        for i in range(len(condition_encodings))
    ]
    conditions = torch.tensor(conditions, dtype=torch.long).T
    conditions_combined = label_encoder(
        tgt_adata_tmp,
        encoder=conditions_combined_encodings,
        condition_key='conditions_combined',
    )
    conditions_combined = torch.tensor(conditions_combined, dtype=torch.long)


In [31]:
decoder_module = InSilicoPerturberGeneration(
    ckpt_masking_path='./T_perturb/T_perturb/tests/'
    'checkpoints/baseline_masking_checkpoint-epoch=00.ckpt',
    ckpt_count_path='./T_perturb/T_perturb/tests/'
    'checkpoints/baseline_counts_checkpoint-epoch=00.ckpt',
    tgt_vocab_size=tgt_vocab_size,
    d_model=d_model,
    num_heads=4,
    num_layers=1,
    d_ff=8,
    max_seq_length=max_seq_length + 10,
    loss_mode='zinb',
    lr=1e-3,
    weight_decay=0.0,
    sequence_length=max_seq_length - 10,
    # lr_scheduler_patience=5.0,
    # lr_scheduler_factor=0.8,
    conditions=conditions_,
    conditions_combined=conditions_combined_,
    n_genes=num_genes,
    dropout=0.0,
    pred_tps=pred_tps,
    context_tps=context_tps,
    n_total_tps=n_total_tps,
    temperature=1.5,
    iterations=19,
    precision='high',
    mask_scheduler='pow',
    output_dir='./T_perturb/T_perturb/tests/res',
    encoder='Transformer_encoder',
    seed=42,
    generate=True,
    var_list=None,
    genes_to_perturb=genes_to_perturb,
    perturbation_token=perturbation_token,
    perturbation_mode=['src', 'tgt'],
)
data_module = CellGenDataModule(
    src_counts=src_counts,
    tgt_counts_dict=tgt_counts_dict,
    src_dataset=src_dataset,
    tgt_datasets=tgt_datasets,
    batch_size=batch_size,
    num_workers=1,
    pred_tps=pred_tps,
    context_tps=context_tps,
    n_total_tps=n_total_tps,
    train_indices=None,
    test_indices=np.random.choice(100, 20, replace=False),
    max_len=max_seq_length,
    condition_keys=condition_keys_,
    condition_encodings=condition_encodings,
    conditions=conditions,
    conditions_combined=conditions_combined,
)
data_module.setup()
test_loader = data_module.test_dataloader()

Using CPU for training
Start datamodule


In [30]:
import importlib
importlib.reload(T_perturb.Model.trainer)
from T_perturb.Model.trainer import InSilicoPerturberGeneration

In [32]:
trainer = pl.Trainer(
    limit_test_batches=1,  # Limit to a single batch for quick testing
    logger=False,
)
trainer.test(decoder_module, data_module)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(limit_test_batches=1)` was configured so 1 batch will be used.


Testing DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]1
perturbating tgt
2
perturbating tgt
perturbating src
perturbation ['src', 'tgt']


100%|██████████| 19/19 [00:00<00:00, 129.51it/s]


true {'tgt_input_ids_t1': tensor([[101,  38,   7,  69,   5,  19,  10,  63,  57,  79,  83,  92,  38,  49,
           0,  92,  56,  68,  80,  78,  97,  51,  97,  87,  45,  97,  97,  93,
          24,  58,  99,  30,  62,  28,  52,  52,   5,  81,  92,  69,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0],
        [101,  78,  28,  32,  19,  68,  10,  71,  49,  38,  92,  22,  95,  83,
           0,  57,  69,  81,  85,  97,  17,  52,  14,  87,  28,  45,  93,  58,
          30,  51,  24,  62,   7,  99,  79,  80,   5,   5,  63,  84,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0],
        [101,  49,  57,  89,  68,  78,  19,  63,  79,  26,   5,  32,  22,  97,
           0,  79,  92,  47,  87,  92,  38,  99,  69,  87,  28,  93,  52,  51,
          79,   7,  24,  10,  83,  30,  14,   5,  31,  62,  80,  17,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0],
        [101,  38,  99,  84,  59,  32,  93,  17,  81,  63,  79,  57,   5,  38,
           0,  49, 

RuntimeError: No active exception to reraise